In [17]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import re
from datetime import datetime
from tqdm import tqdm
import os

# --- 함수 정의 ---

def search_transfermarkt_player_by_name(player_name):
    """오직 선수 이름으로만 Transfermarkt를 검색하여 가장 첫 번째 결과의 ID를 반환합니다."""
    search_url = f"https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36'}
    params = {'query': player_name}
    try:
        response = requests.get(search_url, headers=headers, params=params)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # 검색 결과의 가장 첫 번째 선수 링크를 찾습니다.
        first_result = soup.select_one("div.box table.items > tbody > tr > td.hauptlink > a")
        
        if first_result:
            href = first_result.get('href', '')
            player_id_match = re.search(r'/spieler/(\d+)', href)
            if player_id_match:
                return player_id_match.group(1)
        return None
    except requests.exceptions.RequestException:
        return None

def get_market_value_history(player_id):
    """선수 ID로 몸값 변화 기록 전체를 가져옵니다."""
    api_url = f"https://www.transfermarkt.com/ceapi/marketValueDevelopment/graph/{player_id}"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36'}
    try:
        response = requests.get(api_url, headers=headers)
        response.raise_for_status()
        return response.json().get('list', [])
    except (requests.exceptions.RequestException, ValueError):
        return []

def get_season(date_obj):
    """날짜를 유럽 축구 시즌(예: '2016-2017')으로 변환합니다."""
    year, month = date_obj.year, date_obj.month
    return f"{year}-{year + 1}" if month >= 7 else f"{year - 1}-{year}"

# --- 1단계: 고유 선수 명단(이름만) 추출 및 저장 ---
print("✅ 1단계: 선수 데이터 파일을 읽어 고유 선수 명단을 추출합니다.")
excel_file = 'fotmob_champs.xlsx'
all_players_list = []

if os.path.exists(excel_file):
    print(f"-> '{excel_file}' 엑셀 파일을 발견했습니다. 시트 데이터를 읽습니다.")
    xls = pd.ExcelFile(excel_file)
    sheet_names = [name for name in xls.sheet_names if re.match(r'\d{8}', name)]
    for sheet_name in tqdm(sheet_names, desc="엑셀 시트 읽는 중"):
        df = pd.read_excel(xls, sheet_name=sheet_name)
        if '선수명' in df.columns:
            all_players_list.append(df[['선수명']]) # 이름만 추출
else:
    print(f"-> '{excel_file}'을 찾지 못했습니다. CSV 파일들을 검색합니다.")
    csv_files = [f for f in os.listdir('.') if 'fotmob_champs.xlsx' in f and f.endswith('.csv')]
    for file_path in tqdm(csv_files, desc="CSV 파일 읽는 중"):
        try:
            df = pd.read_csv(file_path)
            if '선수명' in df.columns:
                all_players_list.append(df[['선수명']]) # 이름만 추출
        except Exception as e:
            print(f"-> '{file_path}' 처리 중 오류: {e}")

if not all_players_list:
    print("\n❌ 오류: 선수 데이터가 포함된 엑셀 또는 CSV 파일을 읽지 못했습니다.")
else:
    unique_players_df = pd.concat(all_players_list, ignore_index=True).drop_duplicates(subset=['선수명'])
    unique_players_df = unique_players_df.sort_values(by='선수명').reset_index(drop=True)
    
    unique_players_df.to_csv('unique_players_list_champs.csv', index=False, encoding='utf-8-sig')
    print(f"총 {len(unique_players_df)}명의 고유 선수를 찾았으며, 'unique_players_list_champs.csv' 파일로 (이름순으로) 저장했습니다.")

✅ 1단계: 선수 데이터 파일을 읽어 고유 선수 명단을 추출합니다.
-> 'fotmob_champs.xlsx' 엑셀 파일을 발견했습니다. 시트 데이터를 읽습니다.


엑셀 시트 읽는 중: 100%|██████████| 9/9 [00:00<00:00,  9.07it/s]

총 1178명의 고유 선수를 찾았으며, 'unique_players_list_champs.csv' 파일로 (이름순으로) 저장했습니다.


In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import re
from tqdm import tqdm
import os

def search_transfermarkt_player_by_name(player_name):
    """오직 선수 이름으로만 Transfermarkt를 검색하여 가장 첫 번째 결과의 ID를 반환합니다."""
    search_url = f"https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36'}
    params = {'query': player_name}
    try:
        response = requests.get(search_url, headers=headers, params=params)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # 검색 결과의 가장 첫 번째 선수 링크를 찾습니다.
        first_result = soup.select_one("div.box table.items > tbody > tr > td.hauptlink > a")
        
        if first_result:
            href = first_result.get('href', '')
            player_id_match = re.search(r'/spieler/(\d+)', href)
            if player_id_match:
                return player_id_match.group(1)
        return None
    except requests.exceptions.RequestException as e:
        # tqdm 진행률 바와 겹치지 않도록 print 대신 tqdm.write 사용
        tqdm.write(f"-> '{player_name}' 선수 검색 중 네트워크 오류: {e}")
        return None

# --- 1단계: unique_players_list.csv 파일 읽기 ---
input_filename = 'unique_players_list_premier.csv'

if not os.path.exists(input_filename):
    print(f"❌ 오류: '{input_filename}' 파일이 없습니다. 이 파일을 먼저 생성해주세요.")
else:
    print(f"✅ 1단계: '{input_filename}' 파일에서 선수 명단을 읽어옵니다.")
    unique_players_df = pd.read_csv(input_filename)
    print(f"총 {len(unique_players_df)}명의 선수에 대한 ID 검색을 시작합니다.")

    # --- 2단계: 각 선수의 Transfermarkt ID 검색 및 저장 ---
    print("\n✅ 2단계: 각 선수의 Transfermarkt ID를 검색하여 'transfermarkt_player_ids.csv' 파일로 저장합니다.")
    player_id_data = []
    
    # tqdm을 사용하여 진행률 표시
    for _, row in tqdm(unique_players_df.iterrows(), total=unique_players_df.shape[0], desc="ID 검색 중"):
        player_name = row['선수명']
        player_id = search_transfermarkt_player_by_name(player_name)
        
        if player_id:
            player_id_data.append({'선수명': player_name, 'Transfermarkt_ID': player_id})
        else:
            tqdm.write(f"-> '{player_name}' 선수의 ID를 찾지 못했습니다.")
        
        # 서버에 부담을 주지 않기 위해 요청 사이에 0.5초의 간격을 둡니다.
        time.sleep(0.5)

    # --- 3단계: 최종 결과물 생성 ---
    if player_id_data:
        id_df = pd.DataFrame(player_id_data)
        
        output_filename = 'transfermarkt_player_ids.csv'
        id_df.to_csv(output_filename, index=False, encoding='utf-8-sig')
        
        print("\n[ ID 검색 결과 (상위 10개) ]")
        print(id_df.head(10).to_string())
        print(f"\n\n🎉 모든 작업 완료! 총 {len(id_df)}명의 선수 ID를 찾아 '{output_filename}' 파일에 저장되었습니다.")
    else:
        print("\n처리된 데이터가 없습니다. 선수 검색 결과를 확인해보세요.")

✅ 1단계: 'unique_players_list_premier.csv' 파일에서 선수 명단을 읽어옵니다.
총 879명의 선수에 대한 ID 검색을 시작합니다.

✅ 2단계: 각 선수의 Transfermarkt ID를 검색하여 'transfermarkt_player_ids.csv' 파일로 저장합니다.


ID 검색 중:   0%|          | 0/879 [00:01<?, ?it/s]   

-> 'aaron connolly' 선수의 ID를 찾지 못했습니다.


ID 검색 중:   0%|          | 1/879 [00:02<28:35,  1.95s/it]   

-> 'aaron cresswell' 선수의 ID를 찾지 못했습니다.


ID 검색 중:   0%|          | 2/879 [00:04<21:52,  1.50s/it]   

-> 'aaron hickey' 선수의 ID를 찾지 못했습니다.


ID 검색 중:   0%|          | 3/879 [00:05<27:59,  1.92s/it]


KeyboardInterrupt: 